# Data audit and cleaning 


In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / 'data' / 'raw' / 'Charging_Data.csv').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find data/raw/Charging_Data.csv from the current directory.')

## 1. Raw file audit

In [13]:
raw_path = PROJECT_ROOT / 'data' / 'raw' / 'Charging_Data.csv'
raw = pd.read_csv(raw_path, encoding='cp1252')
print(raw.shape)
raw.dtypes

(441077, 24)


UserID                       int64
Charging Post ID             int64
Location Information           str
District Name                  str
Order creation time            str
Transaction power/kwh      float64
Electricity cost/Yuan      float64
Service charge/Yuan        float64
Transaction Amount/Yuan    float64
Actual Payment/Yuan        float64
Start Time                     str
End Time                       str
Payment time                   str
Temperature(¡æ)            float64
Relative Humidity(%)         int64
Precipitation(mm)          float64
end_cause                      str
is_full_charge               int64
is_user_stop                 int64
is_nonsystem_fault           int64
is_system_issue              int64
is_EV_fault                  int64
is_other_fault               int64
is_abnormal                  int64
dtype: object

In [14]:
raw.isna().sum()

UserID                     0
Charging Post ID           0
Location Information       0
District Name              0
Order creation time        0
Transaction power/kwh      0
Electricity cost/Yuan      0
Service charge/Yuan        0
Transaction Amount/Yuan    0
Actual Payment/Yuan        0
Start Time                 0
End Time                   0
Payment time               0
Temperature(¡æ)            0
Relative Humidity(%)       0
Precipitation(mm)          0
end_cause                  0
is_full_charge             0
is_user_stop               0
is_nonsystem_fault         0
is_system_issue            0
is_EV_fault                0
is_other_fault             0
is_abnormal                0
dtype: int64

In [15]:
raw.head()

,UserID,Charging Post ID,Location Information,District Name,Order creation time,Transaction power/kwh,Electricity cost/Yuan,Service charge/Yuan,Transaction Amount/Yuan,Actual Payment/Yuan,...,Relative Humidity(%),Precipitation(mm),end_cause,is_full_charge,is_user_stop,is_nonsystem_fault,is_system_issue,is_EV_fault,is_other_fault,is_abnormal
0,1,1,Tourist Attraction,Nanhu,2020-12-31 23:35:55\t,6.24,2.60,3.74,6.34,6.34,...,45,0.0,User stops charging,0,1,0,0,0,0,0
1,2,2,Tourist Attraction,Nanhu,2020-12-31 23:35:17\t,0.62,0.26,0.37,0.63,0.63,...,45,0.0,User stops charging,0,1,0,0,0,0,0
2,3,3,Tourist Attraction,Nanhu,2020-12-31 22:01:34\t,28.01,11.66,16.81,28.47,28.47,...,45,0.0,User stops charging,0,1,0,0,0,0,0
3,4,2,Tourist Attraction,Nanhu,2020-12-31 22:21:37\t,5.20,2.17,3.12,5.29,5.29,...,45,0.0,User stops charging,0,1,0,0,0,0,0
4,5,1,Tourist Attraction,Nanhu,2020-12-31 22:10:30\t,4.58,1.91,2.75,4.66,4.66,...,45,0.0,Charging ends normally,1,0,0,0,0,0,0


In [16]:
print(repr(raw['Start Time'].iloc[0]))
print(raw['Location Information'].unique())

'2020-12-31 23:35:43\t'
<StringArray>
[           'Tourist Attraction',     'Financial Industrial Park',
               'Technology Park',                 'Shopping Mall',
                        'Park A',             'Industrial Park\t',
                        'Park B',                   'Bus Station',
             'Government Agency', 'Expressway Service District B',
 'Expressway Service District A', 'Expressway Service District C',
              'Wholesale Market']
Length: 13, dtype: str


## 2. Cardinality checks
Each charger should belong to exactly one location and district.

In [17]:
g = raw.groupby('Charging Post ID').agg(n=('UserID','size'),
        locations=('Location Information','nunique'), districts=('District Name','nunique'))
print('chargers:', len(g), '| max locations per charger:', g['locations'].max(), '| max districts:', g['districts'].max())
raw.groupby(['District Name','Location Information'])['Charging Post ID'].nunique()

chargers: 92 | max locations per charger: 1 | max districts: 1


District Name  Location Information         
Nanhu          Financial Industrial Park        8
               Technology Park                  4
               Tourist Attraction               8
Tongxiang      Bus Station                      8
               Industrial Park\t                8
               Park A                           8
               Park B                           8
               Shopping Mall                    8
Xiuzhou        Expressway Service District A    4
               Expressway Service District B    4
               Expressway Service District C    8
               Government Agency                8
               Wholesale Market                 8
Name: Charging Post ID, dtype: int64

## 3. Run the cleaner

In [ ]:
# Clean the raw data directly in this notebook.
df = raw.copy()

# Remove trailing spaces and tabs from text fields.
for column in df.select_dtypes(include=['object', 'string']).columns:
    df[column] = df[column].str.strip()

# Standardize the columns used by the availability analysis.
df = df.rename(columns={
    column: 'temp_c' for column in df.columns if column.startswith('Temperature')
})
df = df.rename(columns={
    'Charging Post ID': 'charger_id',
    'Location Information': 'location',
    'District Name': 'district',
    'Start Time': 'start',
    'End Time': 'end',
    'Transaction power/kwh': 'energy_kwh',
    'Relative Humidity(%)': 'humidity',
    'Precipitation(mm)': 'precip_mm',
})

# Parse timestamps and remove invalid or non-positive sessions.
for column in ['start', 'end', 'Order creation time', 'Payment time']:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors='coerce')
df = df.dropna(subset=['start', 'end']).copy()
df['duration_min'] = (df['end'] - df['start']).dt.total_seconds() / 60
df = df[df['duration_min'] > 0].drop_duplicates()

# Keep only the first session when sessions overlap on the same charger.
df = df.sort_values(['charger_id', 'start']).reset_index(drop=True)
previous_max_end = df.groupby('charger_id')['end'].transform(lambda end: end.cummax().shift())
df = df[df['start'] >= previous_max_end].reset_index(drop=True)

# Save the cleaned dataset under data/processed/.
output_path = PROJECT_ROOT / 'data' / 'processed' / 'Charging_Data_clean.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f'Saved {len(df):,} cleaned rows to {output_path}')
df.head()

## 4. Validate the cleaned data

In [19]:
assert df['duration_min'].gt(0).all()
assert not df.duplicated().any()
prev_end = df.groupby('charger_id')['end'].shift()
assert not (df['start'] < prev_end).any(), 'overlaps remain'
print('all checks passed')

all checks passed


In [20]:
df['duration_min'].describe(percentiles=[.01,.05,.5,.95,.99]).round(1)

count    440972.0
mean         54.7
std          94.6
min           0.0
1%            0.4
5%            1.1
50%          33.3
95%         150.6
99%         557.5
max        1439.1
Name: duration_min, dtype: float64

In [ ]:
TABLES = PROJECT_ROOT / 'team' / 'mariam_availability' / 'reports' / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)
previous_max_end = df.groupby('charger_id')['end'].transform(lambda end: end.cummax().shift())
audit = pd.Series({
    'raw_rows': len(raw),
    'clean_rows': len(df),
    'removed_rows': len(raw) - len(df),
    'raw_missing_cells': int(raw.isna().sum().sum()),
    'clean_missing_cells': int(df.isna().sum().sum()),
    'nonpositive_durations': int((df['duration_min'] <= 0).sum()),
    'duplicate_rows': int(df.duplicated().sum()),
    'overlaps_with_any_prior_session': int((df['start'] < previous_max_end).sum()),
    'sessions_over_12h': int((df['duration_min'] > 720).sum()),
    'duration_p50_min': float(df['duration_min'].quantile(.50)),
    'duration_p95_min': float(df['duration_min'].quantile(.95)),
    'duration_p99_min': float(df['duration_min'].quantile(.99)),
    'duration_max_min': float(df['duration_min'].max()),
}, name='value').to_frame()
audit.to_csv(TABLES / 'session_audit.csv')
raw_missing = raw.isna().sum().rename('missing_values').to_frame()
raw_missing['missing_pct'] = raw_missing['missing_values'] / len(raw) * 100
clean_missing = df.isna().sum().rename('missing_values').to_frame()
clean_missing['missing_pct'] = clean_missing['missing_values'] / len(df) * 100
raw_missing.to_csv(TABLES / 'missingness_raw.csv')
clean_missing.to_csv(TABLES / 'missingness_clean.csv')
audit

## 5. Columns forbidden as model inputs 
These are only known after the session ends. Keep them out of every availability feature set.

In [21]:
forbidden = ['end','Payment time','energy_kwh','Electricity cost/Yuan','Service charge/Yuan',
             'Transaction Amount/Yuan','Actual Payment/Yuan','end_cause','duration_min',
             'is_full_charge','is_user_stop','is_nonsystem_fault','is_system_issue','is_EV_fault','is_other_fault','is_abnormal']
allowed_base = ['charger_id','location','district','start','temp_c','humidity','precip_mm']
print('allowed base columns:', allowed_base)

allowed base columns: ['charger_id', 'location', 'district', 'start', 'temp_c', 'humidity', 'precip_mm']
